### Import modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager, rc
from pandas import Series, DataFrame
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, cross_validate, KFold, ShuffleSplit
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PowerTransformer, FunctionTransformer, LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectPercentile
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss, mean_squared_error
from category_encoders import TargetEncoder, BinaryEncoder
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.distributions import CategoricalDistribution, IntDistribution, FloatDistribution
from optuna.integration import OptunaSearchCV, ShapleyImportanceEvaluator
from sklearn.cluster import KMeans
from tqdm import tqdm

### Step 1: Data Preprocessing

##### Read data

In [ ]:
X_train = pd.read_csv('../data/X_train.csv', encoding='cp949')
y_train = pd.read_csv('../data/y_train.csv').gender
X_test = pd.read_csv('../data/X_test.csv', encoding='cp949')

# submission을 만들 때 사용하기 위해 ID 저정
id_test = X_test.id

##### Explore data

In [ ]:
# 결측값 존재여부와 범주형 feature 확인
X_train.info()

In [ ]:
# 결측값이 50퍼센트 이상인 칼럼 확인
missing_values = X_train.isnull().mean()
columns_to_drop = missing_values[missing_values >= 0.5].index.tolist()
columns_to_drop

In [ ]:
#결측값이 50퍼센트 이상인 칼럼 삭제
X_train.drop(columns = ['환불금액', '환불건수', '화장품구매주기', '주환불상품',
                        '남성포함상품구매건수', '행사상품구매수', '만족도 떨어지는 제품 총 구입금액'], axis =1)
X_test.drop(columns = ['환불금액', '환불건수', '화장품구매주기', '주환불상품',
                        '남성포함상품구매건수', '행사상품구매수', '만족도 떨어지는 제품 총 구입금액'], axis =1)

In [ ]:
# X_train에서 object 타입의 열(column) 확인하기
object_columns = X_train.select_dtypes(include=['object']).columns
print(object_columns)

In [ ]:
# 범주형 변수와 수치형 변수를 분리
categorical_features = ['주구매상품', '주구매지점', '고객등급', '주환불상품', '라이프 스타일(평균)', '라이프 스타일(건수)', '주구매 요일', '선호방문계절']
numeric_features = list(set(X_train.columns)-set(categorical_features))

##### Impute missing values

In [ ]:
from sklearn.impute import SimpleImputer 

if len(numeric_features) > 0:
    imp = SimpleImputer(strategy='mean')
    X_train[numeric_features] = imp.fit_transform(X_train[numeric_features])
    X_test[numeric_features] = imp.transform(X_test[numeric_features])
if len(categorical_features) > 0:  
    imp = SimpleImputer(strategy="most_frequent")
    X_train[categorical_features] = imp.fit_transform(X_train[categorical_features])
    X_test[categorical_features] = imp.transform(X_test[categorical_features])

In [ ]:
# for i in X_train[categorical_features]:
#     print(X_train[i].value_counts())

In [ ]:
# X_train['주환불상품'].value_counts() #=> 카디날리티가 데이터가 2482개임에비해 너무높음 => 

##### Transform features (Feature Scaling)

In [ ]:
# DNN 모델링에서는 StandardScaler을 주로 사용
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

##### Encode Categorical Variables

In [ ]:
# encoding
for i in tqdm(categorical_features):
    le = LabelEncoder()
    le=le.fit(X_train[i])
    X_train[i]=le.transform(X_train[i])

    for label in np.unique(X_test[i]):
        if label not in le.classes_: 
            le.classes_ = np.append(le.classes_, label)
    X_test[i]=le.transform(X_test[i])

In [ ]:
# # 중요도 1이상인것들끼리 조합의 합을 만듦
# X_train['상위2개합'] = X_train['주환불상품_넥타이(특정)']+X_train['주구매지점_전주점']
# X_train['3등4등합'] = X_train['주환불상품_4대 B/D']+X_train['주환불상품_편집매장']
# X_train['5등6등합'] = X_train['주환불상품_N.B']+X_train['주환불상품_보석']
# X_train['7등8등합'] = X_train['주환불상품_숙녀고정행사']+X_train['주환불상품_스트리트']
# X_train['9등10등합'] = X_train['주구매상품_액세서리']+X_train['주구매상품_침구/수예']



# X_test['상위2개합'] = X_test['주환불상품_넥타이(특정)']+X_test['주구매지점_전주점']
# X_test['3등4등합'] = X_test['주환불상품_4대 B/D']+X_test['주환불상품_편집매장']
# X_test['5등6등합'] = X_test['주환불상품_N.B']+X_test['주환불상품_보석']
# X_test['7등8등합'] = X_test['주환불상품_숙녀고정행사']+X_test['주환불상품_스트리트']
# X_test['9등10등합'] = X_test['주구매상품_액세서리']+X_test['주구매상품_침구/수예']

In [ ]:
# # 1등별 2등 groupby
# X_train=pd.merge(X_train, X_train.groupby('주환불상품_넥타이(특정)')['주구매지점_전주점'].sum().reset_index().rename(columns={'주구매지점_전주점':'주환불상품_넥타이(특정)별 주구매지점_전주점sum'}), on = '주환불상품_넥타이(특정)',how='left')
# X_test=pd.merge(X_test, X_test.groupby('주환불상품_넥타이(특정)')['주구매지점_전주점'].sum().reset_index().rename(columns={'주구매지점_전주점':'주환불상품_넥타이(특정)별 주구매지점_전주점sum'}), on = '주환불상품_넥타이(특정)',how='left')

<font color='tomato'><font color="#CC3D3D"><p>
### Step 2: Modeling
아래 코드는 절대 수정하지 말것!!! 수정하여 submission 제출시 무조건 0점 처리함!!!

# CatBoost

In [ ]:
models_CB = cross_validate(CatBoostClassifier(iterations = 1000, depth = 6, verbose=True, random_state=0),
                        X_train, y_train, 
                        cv=5, 
                        scoring='roc_auc', 
                        return_estimator=True, n_jobs=-1)
oof_pred_CB = np.array([m.predict_proba(X_test)[:,1] for m in models_CB['estimator']]).mean(axis=0)

scores = models_CB['test_score']
print("\nCatBoost CV scores: ", scores)
print("CatBoost CV mean = %.2f" % scores.mean(), "with std = %.2f" % scores.std())

In [ ]:
sub1 = X_test[['id']]
sub1['gender'] = oof_pred_CB
sub1.to_csv('sub1.csv', index=False)

# ExtraTree

In [ ]:
models_ET = cross_validate(ExtraTreesClassifier(n_estimators = 2000, random_state = 0, n_jobs=-1),
                        X_train, y_train, 
                        cv=5, 
                        scoring='roc_auc', 
                        return_estimator=True)
oof_pred_ET = np.array([m.predict_proba(X_test)[:,1] for m in models_ET['estimator']]).mean(axis=0)

scores = models_ET['test_score']
print("ET CV scores: ", scores)
print("ET CV mean = %.2f" % scores.mean(), "with std = %.2f" % scores.std())

criterion{“gini”, “entropy”, “log_loss”}, default=”gini” 바꿔가매해보세요

In [ ]:
#pred_ET = model_ET.fit(X_train, y_train).predict_proba(X_test)[:,1]

In [ ]:
sub2 = X_test[['id']]
sub2['gender'] = oof_pred_ET
sub2.to_csv('sub2.csv', index=False)

# LGBM

In [ ]:
# model_LGBM = LGBMClassifier(learning_rate = 0.01,num_leaves=31,n_estimators=1000, random_state = 0)

In [ ]:
models_LGBM = cross_validate(LGBMClassifier(learning_rate = 0.1,n_estimators=1000,eval_metric='auc' , random_state = 0,n_jobs= -1),
                        X_train, y_train, 
                        cv=5, 
                        scoring='roc_auc', 
                        return_estimator=True)
oof_pred_LGBM = np.array([m.predict_proba(X_test)[:,1] for m in models_LGBM['estimator']]).mean(axis=0)

scores = models_LGBM['test_score']
print("LGBM CV scores: ", scores)
print("LGBM CV mean = %.2f" % scores.mean(), "with std = %.2f" % scores.std())

In [ ]:
#pred_LGBM = model_LGBM.fit(X_train, y_train).predict_proba(X_test)[:,1]

In [ ]:
sub6 = X_test[['id']]
sub6['gender'] = oof_pred_LGBM
sub6.to_csv('sub6.csv', index=False)

---

In [ ]:
weighted_average = []
for i in range(len(oof_pred_CB)):  # 예측값들의 길이가 모두 같다고 가정
    weighted_average.append(oof_pred_CB[i] * 0.6 + oof_pred_ET[i] * 0.3 + oof_pred_LGBM[i] * 0.1)

In [ ]:
final2 = X_test[['id']]
final2['gender'] = weighted_average
final2.to_csv('final2.csv', index=False)

<font color="#CC3D3D"><p>
# End